# Chapter 1 - Lab 3: <font color='blue'>Agentic vs non-Agentic workflow</font>

**<font color='purple'>Goal</font>**:
In this lab, you'll compare **Agentic** and **non-agentiw workflow**:

* Chatting with an LLM without access to external resources
* Chatting with an LLM with augmented information using external APIs:
    -  Fetch Historical Prices for a given stock (The same use case in the book)
    -  Collect News (Additional use case to practice)

**<font color='purple'>Tech stack</font>**:

We'll use :
* OpenAI Responses API ==> Standard calls + Function Callings
* OpenAI Agent SDK ==> Simple Agent Abstraction
* NewsApi: https://newsapi.org/

You need to create API Keys in both OpenAI (you need to add some credits) and NewsAPI (free until a certain # of calls).

Knowledge Cutoff of GPT-4.1 is June 2024

## Install packages

Install openai and newsapi packages

In [ ]:
%pip install openai -q
%pip install newsapi-python -q

Add you OpenAI Key in Google colab: in the left vertical menu, you have an icon key. Click on it, and add your OpenAI key with a name

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

NEWS_API_KEY = userdata.get('NEWS_API_KEY')

To have more information on how to call the API, you can read the documentation below.

In the various labs, you'll have a detailed overview on how to use the API.

https://platform.openai.com/docs/api-reference



# 1- Non Agentic Workflow

## Historical Price

### Responses API - GPT-4.1

https://platform.openai.com/docs/api-reference/responses

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1",
  input="What is the last price of NVIDIA."
)

print(response)

Response(id='resp_684ed16983ec8199a7b06e83d7803c6c0fc5d337140b8d21', created_at=1749995881.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_684ed16a64a881998ed0965d545b9a370fc5d337140b8d21', content=[ResponseOutputText(annotations=[], text='As of my latest update in June 2024, I do not have real-time data access. To find the most **recent price of NVIDIA (NVDA)** stock, please check a reliable financial news source or stock market platform such as:\n\n- [Yahoo Finance: NVDA](https://finance.yahoo.com/quote/NVDA)\n- [Google Finance: NVDA](https://www.google.com/finance/quote/NVDA:NASDAQ)\n- [NASDAQ: NVDA Quote](https://www.nasdaq.com/market-activity/stocks/nvda)\n\nThese sites provide the latest market prices, charts, and news. If you need historical price details or information up until a certain date, let me know!', type='output_text', logprobs=None)], role='assistant', status='

In [ ]:
print(response.output[0].content[0].text)

As of my latest update in June 2024, I do not have real-time data access. To find the most **recent price of NVIDIA (NVDA)** stock, please check a reliable financial news source or stock market platform such as:

- [Yahoo Finance: NVDA](https://finance.yahoo.com/quote/NVDA)
- [Google Finance: NVDA](https://www.google.com/finance/quote/NVDA:NASDAQ)
- [NASDAQ: NVDA Quote](https://www.nasdaq.com/market-activity/stocks/nvda)

These sites provide the latest market prices, charts, and news. If you need historical price details or information up until a certain date, let me know!


### Results

As yu can see, the LLM was not able to provide the last price of NVIDIA, as its knwoledge cufoff was on June 2024.

### Another Try

In this part, I'll provide a web site link. let's see what happens:

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response2 = client.responses.create(
  model="gpt-4.1",
  input="What is the last price of NVIDIA. Can you access to https://finance.yahoo.com/quote/NVDA to collect the last price? "
)

print(response2.output[0].content[0].text)

I can't directly access or browse external websites, such as [Yahoo Finance](https://finance.yahoo.com/quote/NVDA), in real time. However, you can check the current price of NVIDIA (ticker: NVDA) by visiting that [Yahoo Finance link](https://finance.yahoo.com/quote/NVDA) or using financial news platforms like [Google Finance](https://www.google.com/finance/quote/NVDA:NASDAQ).

If you’d like, I can guide you on how to find this information quickly, or help you interpret latest price data if you paste it here! Let me know how I can assist further.


I have the same issue, the LLM cannot provide the price because it cannot access and browse external websites like this.

## News

### Responses API

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1",
  input="Give me the latest news about NVIDIA."
)

print(response.output[0].content[0].text)

Here are some of the latest news highlights about **NVIDIA** as of June 2024:

---

### 1. **NVIDIA Surpasses $3 Trillion Market Cap**
- NVIDIA recently became the world’s second most valuable company, surpassing Apple with a market capitalization above $3 trillion. This surge is driven by ongoing demand for NVIDIA's AI and data center chips.

### 2. **NVIDIA Unveils Next-Gen Rubin GPUs**
- At Computex 2024, NVIDIA revealed its new **Rubin GPU architecture**, which is expected to power its next line of AI accelerators and graphics cards. This follows their highly successful Blackwell GPUs.

### 3. **AI and Data Center Growth**
- NVIDIA's Q1 2024 earnings report highlights explosive growth in its data center segment, mainly due to soaring demand for AI hardware among companies building large language models and generative AI services.

### 4. **Partnerships with Major Tech Firms**
- NVIDIA is expanding partnerships with companies like Microsoft, Amazon, and Google to provide AI infrastr

### Results

If you search for "NVIDIA Surpasses $3 Trillion Market Cap" in the web, you'll see that this news is from June 2024... as it's mentionned by the llm itself in the output.

# 2- Agentic Workflow:

In the following implementation, we'll use tools call to avoid knowledge cutoff and therefore build a basic agentic systeme

## Tools Calling without agent abstraction: Historical

Here, you'll learn how to use function calling with Responses API, without using any agent abstraction

1. You need to define what we call "Tools" object: Type (function), name, Description, parameters...
2. Declare your function: Tool to access external API
3. Call Responses API by enabling: "tools"

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

In [ ]:
import yfinance as yf

In [ ]:
from openai import OpenAI
import json

client = OpenAI(api_key = OPENAI_API_KEY)

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_latest_stock_price",
        "description": "Fetch the current stock price for the given symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "A ticker symbol of a given stock",
                },
            },
            "required": ["symbol"],
        },
    },
]

# Define the tool
def get_latest_stock_price(symbol: str) -> dict():
    """
    Fetch the current stock price for the given symbol.
    """

    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="1d")
    latest_close = hist['Close'].iloc[-1]
    latest_date = hist.index[-1].strftime('%Y-%m-%d')

    return {'price': latest_close, 'date': latest_date}


input_list = [
    {"role": "user", "content": "What is the current price of nvidia."}
]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-4.1-mini",
    tools=tools,
    input=input_list,
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_IOWQn2fGCRnssvy0WYOfx0Ab', name='get_latest_stock_price', type='function_call', id='fc_08b64f5293e3724c0068fbe05a1c3c8193985b28200816858c', status='completed')]


In [ ]:
response.output

[ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_oxtqNG73OHP89OL4oNjE6NZ8', name='get_latest_stock_price', type='function_call', id='fc_09b9ecc23188c1760068fbde293e548194ad250fe69afd9bab', status='completed')]

* This will generate a ResponseFunctionToolCall with the name of the function, and the argument (well interpreted by the model, it gives a symbol instead of the name of the company)
* We then to execute this function with the right argument


In [ ]:
# Save function call outputs for subsequent requests
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_latest_stock_price":
            print(json.loads(item.arguments))
            # 3. Execute the function logic
            prices = get_latest_stock_price(json.loads(item.arguments)['symbol'])
            print(prices)

            # 4. Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "prices": prices
                })
            })

print("Final input:")
print(input_list)

{'symbol': 'NVDA'}
{'price': np.float64(186.25999450683594), 'date': '2025-10-24'}
Final input:
[{'role': 'user', 'content': 'What is the current price of nvidia.'}, ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_IOWQn2fGCRnssvy0WYOfx0Ab', name='get_latest_stock_price', type='function_call', id='fc_08b64f5293e3724c0068fbe05a1c3c8193985b28200816858c', status='completed'), {'type': 'function_call_output', 'call_id': 'call_IOWQn2fGCRnssvy0WYOfx0Ab', 'output': '{"prices": {"price": 186.25999450683594, "date": "2025-10-24"}}'}]


* Send back the whole result to the LLM to interpret and output the final answer

In [ ]:
# Call the API with the updated input list
response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="Respond only with price and date generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final output:
{
  "id": "resp_08b64f5293e3724c0068fbe09aa4848193b6e0ae5ce0c75ffe",
  "created_at": 1761337498.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with price and date generated by a tool.",
  "metadata": {},
  "model": "gpt-4.1-mini-2025-04-14",
  "object": "response",
  "output": [
    {
      "id": "msg_08b64f5293e3724c0068fbe09c5964819387cdb34070772b0e",
      "content": [
        {
          "annotations": [],
          "text": "The current price of Nvidia (NVDA) is $186.26 as of October 24, 2025.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_latest_stock_price",
      "parameters": {
        "type": "object",
        "properties": {
          "symbol": {
            "type": "string",
            "desc

https://composio.dev/blog/claude-function-calling-tools

## OpenAI Agent SDK

In this part, we'll be using OpenAI Agent SDK.

You need first to install the package.

We'll create:
*  A tool: to fetch historical prices
*  An agent:
   - Describe the instructions to give to the agent
   - List of the tools the agent can use
   - Model

Install OpenAI Agents SDK

In [ ]:
!pip install openai-agents -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.4/144.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.8 MB/s eta 0:00:00


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

#We import the agent components
from agents import Agent, Runner, function_tool
import nest_asyncio
nest_asyncio.apply()

### Historical Prices Agent

#### Tool

➡ Create a tool to fetch historical prices

In [ ]:
import yfinance as yf


@function_tool
def get_latest_stock_price(symbol: str) -> dict():
    """
    Fetch the current stock price for the given symbol.
    """

    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="1d") # Fetch the last 1 day of historical data
    latest_close = hist['Close'].iloc[-1]
    latest_date = hist.index[-1]

    return {'price': latest_close, 'date': latest_date}

In [ ]:
ticker = yf.Ticker("NVDA")
hist = ticker.history(period="1d")
hist

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-08-15 00:00:00-04:00,181.759995,181.899994,178.044693,178.919998,65930518,0.0,0.0


#### Agent

##### Define your agent

In [ ]:
MODEL = "gpt-4.1"
stock_price_agent = Agent(
    name="Stock Price Agent",
    instructions="Fetch Historical prices for a given stock symbol",
    model = MODEL,
    tools=[get_latest_stock_price],
)

##### Run your agent

In [ ]:
output = Runner.run_sync(
    starting_agent=stock_price_agent,
    input="Which date is to date and What is the current stock price of Nvidia?",
)

print(output.final_output)

Today's date for the latest price data is August 15, 2025.

The current stock price of Nvidia (NVDA) is $178.89.


In [ ]:
output = Runner.run_sync(
    starting_agent=stock_price_agent,
    input="Which date is to date and What is the current stock price of Nvidia?",
)

print(output.final_output)

### News Agent

In [ ]:
from newsapi import NewsApiClient
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

from datetime import date, timedelta

#### Tool

Here I defined two tools:
* One fetching latest news from NewsAPI ==> **search_news**
* The other is summarizing these news, using an LLM ==> **summarize_news_api**

In [ ]:
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

Response(id='resp_03379e1e930c147a0068fce52110fc81969249b8397d011b96', created_at=1761404193.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_03379e1e930c147a0068fce52233088196a7abdb2672aa2e42', content=[ResponseOutputText(annotations=[], text='As of the **market close on June 14, 2024**, the last price of **NVIDIA Corporation (NVDA)** was **$130.98 USD** per share.\n\n*Please note*: Stock prices are subject to change during market hours. For the most current price, please check a reliable financial news source or your brokerage account.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, reasoning=Reasoning(effort

In [ ]:
END_DATE = date.today()
START_DATE = date.today()-timedelta(days=7)


def _search_news_internal(ticker: str, num_articles:int = 3, to_datetime:date = END_DATE,from_datetime:date = START_DATE) -> str:
  """
  Internal function that get the most recent news of a stock or an instrument

  Args:
  ticker (str): the stock ticker to be given to NEWSAPI
  num_articles (int): Number of news article to collect
  to_datetime (date): Today's date
  from_datetime (date): Date from which to collect the news (7 days before today for example)
  """



  all_articles = newsapi.get_everything(q=ticker,
                                            from_param=from_datetime,
                                            to=to_datetime,
                                            language='en',
                                            sort_by='relevancy',
                                            page_size=num_articles)

  news_concat =  [
    f"{article['title']}, {article['description']}, {article['publishedAt']},{article['content'][0:100]}"
    for article in all_articles['articles']
    ]

  return (".\n").join(news_concat)


@function_tool
def search_news(ticker: str, num_articles:int = 3, to_datetime:date = END_DATE,from_datetime:date = START_DATE) -> str:
  """
  Get the most recent news of a stock or an instrument

  Args:
  ticker (str): the stock ticker to be given to NEWSAPI
  num_articles (int): Number of news article to collect
  to_datetime (date): Today's date
  from_datetime (date): Date from which to collect the news (7 days before today for example)
  """
  return _search_news_internal(ticker, num_articles, to_datetime,from_datetime)

In [ ]:

END_DATE = date.today()
START_DATE = date.today()-timedelta(days=7)

@function_tool
def summarize_news_api(ticker: str,num_articles:int = 3, to_datetime:date = END_DATE,from_datetime:date = START_DATE) -> str:
  """
  Summarize the news of a given stock or an instrument

  Args:
    ticker (str): the stock ticker to be given to NEWSAPI
    num_articles (int): Number of news article to collect
    to_datetime (date): Today's date
    from_datetime (date): Date from which to collect the news (7 days before today for example)

  """

  news = _search_news_internal(ticker, num_articles, to_datetime,from_datetime)

  prompt = f"Summarize the following text by extracting the key insights: {news}"

  response = client.responses.create(
  model="gpt-4.1",
  input= prompt
  )

  return response.output[0].content[0].text

#### Running NewsAPI itself without LLM/Agent

In [ ]:
num_articles = 3
to_datetime = END_DATE
from_datetime = START_DATE
ticker = " NVDA"

all_articles = newsapi.get_everything(q=ticker,
                                            from_param=from_datetime,
                                            to=to_datetime,
                                            language='en',
                                            sort_by='relevancy',
                                            page_size=num_articles)

news_concat =  [
    f"{article['title']}, {article['description']}, {article['publishedAt']},{article['content'][0:100]}"
    for article in all_articles['articles']
    ]

news = (".\n").join(news_concat)

prompt = f"Summarize the following text by extracting the key insights: {news}"
response = client.responses.create(
  model="gpt-4.1",
  input= prompt
  )

print(response.output[0].content[0].text)

Certainly! Here are the **key insights extracted** from the text:

- **Nvidia's Dominance in S&P 500:** Nvidia now represents 8% of the S&P 500 Index, meaning the stock’s movements heavily influence overall market performance, raising concerns about overconcentration and increased market risk.
- **Historical Parallels:** The current dominance by Nvidia is reminiscent of past periods when other sectors or companies, such as energy stocks and Exxon Mobil, held outsized influence over the index.
- **Growth Driven by AI:** Nvidia’s rapid rise is fueled by its leadership in artificial intelligence (AI) technologies.
- **Bullish Projections:** There is speculation about whether Nvidia’s stock can surpass $300 by the end of 2025, depending on sustained AI momentum.
- **Strategic Partnerships and Achievements:** Nvidia and Taiwan Semiconductor Manufacturing (TSMC) have produced the first Blackwell wafer in the US, showcasing Nvidia’s continued innovation and manufacturing reach.
- **Strong Rev

In [ ]:
news

'Nvidia Stock at 8% of the S&P 500 Index Is a Big Problem for Investors. Let’s Do the Math., One single stock essentially dictates our financial lives these days., 2025-10-21T14:42:22Z,Remember when energy stocks dominated the S&amp;P 500 Index ($SPX), and one company, Exxon Mobil (XO.\nCan Nvidia Stock Hit $300 in 2025?, Will Nvidia’s unstoppable AI momentum drive the stock past $300 by the end of 2025?, 2025-10-22T13:00:02Z,When it comes to artificial intelligence (AI), one name that towers above the rest is Nvidia (NVDA)..\nNVIDIA Corporation (NVDA) And Taiwan Semiconductor Manufacturing (TSM) Produces the First Blackwell Wafer in US, \u200bNVIDIA Corporation (NASDAQ:NVDA) is one of the Best Revenue Growth Stocks to Invest In. On October 17, NVIDIA Corporation (NASDAQ:NVDA) announced reaching a..., 2025-10-22T09:17:19Z,NVIDIA Corporation (NASDAQ:NVDA) is one of the Best Revenue Growth Stocks to Invest In. On October 1'

In [ ]:
#Another call to see if I have a consistent output
prompt = f"Summarize the following text by extracting the key insights: {news}"
response = client.responses.create(
  model="gpt-4.1",
  input= prompt
  )

In [ ]:
print(response.output[0].content[0].text)

Here are the **key insights** from the provided text:

1. **Nvidia's Dominance**: Nvidia’s stock now makes up about 8% of the S&P 500 Index, giving it outsized influence on the index and investors’ portfolios, similar to how Exxon Mobil once dominated.

2. **Concentration Risk**: The high weighting of a single stock (Nvidia) is a potential problem for investors, as the performance of one company can disproportionately impact overall investment returns.

3. **AI Momentum**: Nvidia’s leadership in artificial intelligence (AI) has driven significant stock gains, with ongoing speculation about whether its momentum could push the stock past $300 by the end of 2025.

4. **Technological Achievements**: Nvidia, in collaboration with Taiwan Semiconductor Manufacturing (TSMC), produced the first Blackwell wafer in the US, marking a milestone in semiconductor technology.

5. **Revenue Growth**: Nvidia is highlighted as one of the best revenue growth stocks, further solidifying its position as a m

#### Agent: Fetch News

In [ ]:
from agents import Agent, Runner, function_tool

In [ ]:
MODEL = "gpt-4o-mini"
news_agent = Agent(
    name="News Agent",
    instructions="Get the latest news of a stock or an instrument using only the tools associated with the agent",
    model = MODEL,
    tools=[search_news],
)

In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Give me the latest news about Apple.",
)

print(output.final_output)

Here are the latest news highlights about Apple:

1. **All-Time High Stock Price**:
   - Apple's stock price has recently hit an all-time high, currently sitting around $263 per share. This surge is attributed to soaring sales of the iPhone 17. *(Date: October 20, 2025)*

2. **Massive Returns for Stockholders**:
   - Over the past ten years, Apple stock has returned an incredible $847 billion to its investors through cash dividends and buybacks. *(Date: October 22, 2025)*

3. **Leading Earnings Reports**:
   - Apple is set to be a key player among several megacap hyperscalers reporting earnings, including Meta Platforms, Alphabet, Amazon, and Microsoft. Its stock remains close to a new buy point, showing positive early signs. *(Date: October 24, 2025)*

If you need more details or additional information, let me know!


In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="give me the latest news about Nvidia?",
)

print(output.final_output)

Here are the latest news articles about Nvidia (NVDA):

1. **Nvidia Stock at 8% of the S&P 500 Index Is a Big Problem for Investors. Let’s Do the Math.**  
   Published on: October 21, 2025  
   Summary: This article discusses how one single stock, Nvidia, significantly influences financial markets, much like how energy stocks once dominated the S&P 500 Index.

2. **Can Nvidia Stock Hit $300 in 2025?**  
   Published on: October 22, 2025  
   Summary: The article explores the potential for Nvidia’s stock to reach $300 by the end of 2025, fueled by its momentum in the artificial intelligence sector.

3. **NVIDIA Corporation (NVDA) And Taiwan Semiconductor Manufacturing (TSM) Produces the First Blackwell Wafer in US**  
   Published on: October 22, 2025  
   Summary: Nvidia announces its partnership with TSM to produce the first Blackwell wafer in the US, highlighting its growth as a leading revenue-generating stock.

If you need more information or specific articles, feel free to ask!


In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Provide the latest news about Apple and Nvidia.",
)

print(output.final_output)

### Latest News on Apple (AAPL)

1. **Apple’s Stock Hits New All-Time High**  
   - **Date**: October 20, 2025  
   - Apple’s stock price recently reached an all-time high of approximately $263 per share, driven by surging iPhone 17 sales.

2. **Apple Stockholders Hit $850 Billion Jackpot**  
   - **Date**: October 22, 2025  
   - Over the past ten years, Apple stock has returned a substantial $847 billion to investors through dividends and buybacks.

3. **Apple Shines in Megacap Earnings Reports**  
   - **Date**: October 24, 2025  
   - Apple is part of a lineup of megacap hyperscalers making headlines in the latest earnings reports, holding near a fresh buy point.

### Latest News on Nvidia (NVDA)

1. **Nvidia Stock at 8% of the S&P 500 Index**  
   - **Date**: October 21, 2025  
   - The proportion of Nvidia stock in the S&P 500 Index poses challenges for investors, highlighting its significant market influence.

2. **Can Nvidia Stock Hit $300 in 2025?**  
   - **Date**: October 22

#### Agent: Summarize news

In [ ]:
MODEL = "gpt-4o-mini"
news_agent = Agent(
    name="News Agent",
    instructions="Get a summary about the latest news of a stock or an instrument using only the tools associated with the agent.",
    # instructions="Give me a summary about the latest news about OpenAI?",
    model = MODEL,
    tools=[summarize_news_api],
)

In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Summarize the latest news of Apple?",
)

print(output.final_output)

Here are the latest insights regarding Apple (AAPL):

- **Stock Performance**: Apple’s stock has reached an all-time high of around $263 per share, attributed to strong sales of the iPhone 17.
  
- **Investor Returns**: Over the last decade, Apple has returned about $847 billion to investors through dividends and stock buybacks.

- **Analyst Ratings**: Wells Fargo has raised its price target for Apple to $290, reaffirming an “Overweight” rating and marking it as one of the top investment options among Fortune 500 companies as of late 2025.


In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Summarize the latest news of Apple?",
)

print(output.final_output)

Here are the latest highlights regarding Apple:

1. **Stock Performance**: Apple's stock has recently hit a record high, trading around $263 per share, with upward momentum continuing.

2. **Product Success**: The strong sales of the iPhone 17 are significantly contributing to the surge in Apple's stock price.

3. **Shareholder Returns**: Over the past decade, Apple has returned an impressive $847 billion to its shareholders through a combination of dividends and stock buybacks.

4. **Analyst Upgrades**: Wells Fargo has raised its price target for Apple to $290 per share, maintaining an 'Overweight' rating. This positions Apple as one of the top stocks in the Fortune 500 for investment opportunities. 

These developments indicate strong market confidence and a successful product lineup for Apple.


In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Give me a summary of the latest news of NVIDIA?",
)

print(output.final_output)

Here's a summary of the latest news regarding NVIDIA:

1. **Market Dominance**: Nvidia now makes up about 8% of the S&P 500 Index, raising systemic risks similar to when Exxon Mobil dominated the index. This concentration means the index's performance is heavily influenced by Nvidia, which could lead to volatility.

2. **AI-Driven Growth Expectations**: As a leader in the AI sector, there's speculation that Nvidia’s stock might exceed $300 by the end of 2025. The company's strong momentum in AI is crucial for its revenue and stock performance.

3. **Technological Advancements**: Nvidia, in collaboration with Taiwan Semiconductor Manufacturing Company (TSMC), has created the first Blackwell wafer in the US, highlighting its innovation and commitment to US manufacturing.

4. **Revenue Growth Leadership**: Nvidia continues to be recognized for exceptional revenue growth, with strong results and technological advancements reported as of October 2025.

Overall, while Nvidia's concentration 

In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Provide a summary of the latest news of NVIDIA.",
)

print(output.final_output)

**Key Insights on NVIDIA:**

- **Market Dominance:** NVIDIA constitutes about 8% of the S&P 500, highlighting concentration risks for index investors.
- **AI Leadership:** The company is at the forefront of the AI sector, with projections suggesting its stock price might exceed $300 by the end of 2025.
- **Technological Milestones:** NVIDIA, in collaboration with TSMC, has successfully produced its first Blackwell wafer in the U.S.
- **Strong Revenue Growth:** Noted as a top revenue growth stock, NVIDIA remains an appealing choice for investors, despite potential risks from significant index concentration.
